<a href="https://colab.research.google.com/github/limarceloaugusto133/lia1_2026_2/blob/main/NutriVision_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥗 NutriVision — Identificação de Alimentos com YOLO + Streamlit

Protótipo para Google Colab: detecta alimentos (com ou sem embalagem) via **Ultralytics YOLO**
e retorna informações nutricionais (calorias, macro e micronutrientes) numa interface **Streamlit**.


## 1. Setup — instalação de dependências

In [ ]:
# Instala as bibliotecas necessárias no ambiente Colab
!pip install -q ultralytics streamlit pyngrok requests opencv-python-headless pillow pandas

# Túnel público simples para servir o Streamlit dentro do Colab
!npm install -g localtunnel --silent


In [ ]:
import os
import sqlite3
import json
from pathlib import Path

# Pastas do projeto
PROJECT_DIR = Path("/content/nutrivision")
MODELS_DIR = PROJECT_DIR / "models"
DATA_DIR = PROJECT_DIR / "data"
DB_PATH = PROJECT_DIR / "nutrition_cache.db"

for d in (PROJECT_DIR, MODELS_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Estrutura de pastas criada em:", PROJECT_DIR)


Estrutura de pastas criada em: /content/nutrivision


## 2. Datasets utilizados para alimentos de supermercado


| Dataset | Conteúdo |
|---|---|
| **Open Images Dataset V7 (subset "Food")** | Milhares de imagens anotadas com bounding boxes de categorias de alimentos (fruta, pão, sanduíche, etc.) |
| **Food-101** | 101 categorias de pratos prontos, aproximadamente 1.000 imagens cada | Bom para classificação, mas **sem bounding boxes** (precisa gerar anotações) |
| **Roboflow Universe — "Grocery Dataset" / "Freiburg Groceries"** | Produtos de supermercado embalados (rótulos, caixas, latas)|
| **Open Food Facts (imagens + dados nutricionais)** | Fotos de embalagens + dados nutricionais oficiais dos produtos |
| **Kaggle — "Grocery Store Dataset" / "Fruits-360"** | Frutas e vegetais isolados, boa qualidade de imagem |





### 2.1 Combinando os 5 datasets

In [ ]:
DATASET_CONFIRMADO = True
CAMINHO_DATA_YAML = str(DATA_DIR / "merged" / "data.yaml")
print("✅ Dataset confirmado. Seguindo para o download e preparação dos 5 datasets.")

✅ Dataset confirmado. Seguindo para o download e preparação dos 5 datasets.


### 2.2 Download — Open Images V7 (subset de alimentos)

Usa o [FiftyOne](https://docs.voxel51.com/integrations/open_images.html) para baixar as
classes de comida do Open Images V7 e exporta
automaticamente em formato YOLO.

In [ ]:
!pip install -q fiftyone


In [ ]:

import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.utils.yolo as fouy

# Classes de alimentos disponíveis no Open Images V7
CLASSES_ALIMENTOS_OI = [
    "Apple", "Banana", "Orange", "Grape", "Strawberry", "Lemon", "Pear", "Peach",
    "Tomato", "Carrot", "Broccoli", "Potato", "Cucumber", "Bell pepper",
    "Bread", "Croissant", "Bagel", "Pretzel", "Muffin", "Cookie", "Pastry",
    "Egg", "Cheese", "Milk", "Yogurt", "Butter",
    "Pizza", "Hamburger", "Hot dog", "Sandwich", "Sushi", "Taco", "Burrito",
    "Pasta", "Guacamole", "Popcorn", "French fries",
    "Cake", "Ice cream", "Doughnut", "Chocolate",
    "Coffee", "Tea", "Juice", "Wine", "Beer",
]

dataset_oi = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=CLASSES_ALIMENTOS_OI,
    max_samples=6000,
    seed=51,
    shuffle=True,
    dataset_name="nutrivision-openimages-food",
)

dataset_oi_val = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=CLASSES_ALIMENTOS_OI,
    max_samples=800,
    seed=51,
    shuffle=True,
    dataset_name="nutrivision-openimages-food-val",
)

OI_EXPORT_DIR = DATA_DIR / "openimages_food"
dataset_oi.export(
    export_dir=str(OI_EXPORT_DIR / "train"),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    classes=CLASSES_ALIMENTOS_OI,
)
dataset_oi_val.export(
    export_dir=str(OI_EXPORT_DIR / "val"),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    classes=CLASSES_ALIMENTOS_OI,
)
print("Open Images (subset food) exportado para:", OI_EXPORT_DIR)


### 2.3 Download — Grocery Store Dataset & Groceries (Roboflow)


In [ ]:
!pip install -q roboflow
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "key")

if ROBOFLOW_API_KEY == "key":
else:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    # Grocery Store Dataset — https://universe.roboflow.com/yolo-9co33/grocery-store-hr9us
    projeto_grocery = rf.workspace("yolo-9co33").project("grocery-store-hr9us")
    versao_grocery = projeto_grocery.version(2)
    versao_grocery.download("yolov8", location=str(DATA_DIR / "grocery_store"))

    # Freiburg Groceries (versão YOLO) — https://universe.roboflow.com/michael-ringer/freiburg-groceries
    projeto_freiburg = rf.workspace("michael-ringer").project("freiburg-groceries")
    versao_freiburg = projeto_freiburg.version(1)
    versao_freiburg.download("yolov8", location=str(DATA_DIR / "freiburg_groceries"))

    print("Grocery Store + Freiburg Groceries baixados em formato YOLO.")


### 2.4 Download — Food-101 e Fruits-360 (classificação → bbox de imagem inteira)

Esses dois não têm bounding boxes. Baixamos como classificação e convertemos cada imagem
numa única caixa cobrindo a imagem inteira, no formato de rótulo YOLO (`classe x_centro
y_centro largura altura`, tudo normalizado 0–1, ou seja `0.5 0.5 1.0 1.0`).


In [ ]:
import torchvision
from torchvision.datasets import Food101
from PIL import Image
import shutil

def converter_classificacao_para_yolo_bbox(pares_imagem_classe, nomes_classes, pasta_saida, prefixo, max_por_classe=80):
    """Recebe uma lista de (caminho_imagem, indice_classe) e grava no formato YOLO
    com uma única caixa cobrindo a imagem inteira (rótulo fraco)."""
    pasta_img = Path(pasta_saida) / "images"
    pasta_lbl = Path(pasta_saida) / "labels"
    pasta_img.mkdir(parents=True, exist_ok=True)
    pasta_lbl.mkdir(parents=True, exist_ok=True)

    contagem_por_classe = {}
    for caminho_origem, idx_classe in pares_imagem_classe:
        contagem_por_classe.setdefault(idx_classe, 0)
        if contagem_por_classe[idx_classe] >= max_por_classe:
            continue
        contagem_por_classe[idx_classe] += 1
        nome_arquivo = f"{prefixo}_{idx_classe}_{contagem_por_classe[idx_classe]}.jpg"
        shutil.copy(caminho_origem, pasta_img / nome_arquivo)
        with open(pasta_lbl / nome_arquivo.replace(".jpg", ".txt"), "w") as f:
            f.write(f"{idx_classe} 0.5 0.5 1.0 1.0\n")  # bbox = imagem inteira

    print(f"{prefixo}: {sum(contagem_por_classe.values())} imagens convertidas em {pasta_saida}")


# --- Food-101 ---
food101_raw = Food101(root=str(DATA_DIR / "food101_raw"), split="train", download=True)
pares_food101 = [(str(food101_raw._image_files[i]), food101_raw._labels[i])
                  for i in range(len(food101_raw))]
converter_classificacao_para_yolo_bbox(
    pares_food101, food101_raw.classes,
    pasta_saida=DATA_DIR / "food101_yolo", prefixo="food101", max_por_classe=60,
)

# --- Fruits-360 (via kagglehub — pede login/API key do Kaggle na primeira vez) ---
!pip install -q kagglehub
import kagglehub
caminho_fruits360 = kagglehub.dataset_download("moltean/fruits")
print("Fruits-360 baixado em:", caminho_fruits360)

pasta_treino_fruits = next(Path(caminho_fruits360).rglob("Training"))
classes_fruits = sorted([d.name for d in pasta_treino_fruits.iterdir() if d.is_dir()])
pares_fruits360 = []
for idx, nome_classe in enumerate(classes_fruits):
    for img_path in (pasta_treino_fruits / nome_classe).glob("*.jpg"):
        pares_fruits360.append((str(img_path), idx))

converter_classificacao_para_yolo_bbox(
    pares_fruits360, classes_fruits,
    pasta_saida=DATA_DIR / "fruits360_yolo", prefixo="fruits360", max_por_classe=40,
)


### 2.5 Merge — combinar os 5 datasets num único conjunto YOLO

Junta todas as pastas de imagens/labels num só lugar (`data/merged`), unifica a lista de
classes (removendo duplicatas de nomes parecidos) e gera o `data.yaml` final usado no
treinamento.


In [ ]:
import yaml

MERGED_DIR = DATA_DIR / "merged"
(MERGED_DIR / "images" / "train").mkdir(parents=True, exist_ok=True)
(MERGED_DIR / "images" / "val").mkdir(parents=True, exist_ok=True)
(MERGED_DIR / "labels" / "train").mkdir(parents=True, exist_ok=True)
(MERGED_DIR / "labels" / "val").mkdir(parents=True, exist_ok=True)

# Fontes: (pasta_imagens, pasta_labels, lista_de_classes_originais, é_validação)
fontes = []

def registrar_fonte_yolo_padrao(pasta_base, nomes_classes, val_split_ratio=0.1):
    """Para datasets já organizados como images/ + labels/ (uma pasta só, sem split)."""
    imgs = sorted((Path(pasta_base) / "images").glob("*.*"))
    n_val = max(1, int(len(imgs) * val_split_ratio))
    return [(imgs[n_val:], "train", nomes_classes, pasta_base), (imgs[:n_val], "val", nomes_classes, pasta_base)]

# Classes combinadas (união, na ordem em que aparecem)
classes_finais = []
mapa_indice = {}  # (dataset_tag, indice_antigo) -> indice_novo

def obter_indice_final(nome_classe):
    nome_norm = nome_classe.strip().lower().replace(" ", "_")
    if nome_norm not in mapa_indice:
        mapa_indice[nome_norm] = len(classes_finais)
        classes_finais.append(nome_norm)
    return mapa_indice[nome_norm]

def copiar_e_remapear(lista_imgs, split, nomes_classes_originais, pasta_labels_origem):
    for img_path in lista_imgs:
        label_path = Path(pasta_labels_origem) / "labels" / (img_path.stem + ".txt")
        if not label_path.exists():
            continue
        novo_nome = f"{pasta_labels_origem.name}_{img_path.name}"
        shutil.copy(img_path, MERGED_DIR / "images" / split / novo_nome)

        linhas_novas = []
        for linha in label_path.read_text().strip().splitlines():
            partes = linha.split()
            idx_antigo = int(partes[0])
            nome_classe = nomes_classes_originais[idx_antigo]
            idx_novo = obter_indice_final(nome_classe)
            linhas_novas.append(" ".join([str(idx_novo)] + partes[1:]))
        (MERGED_DIR / "labels" / split / novo_nome.replace(Path(novo_nome).suffix, ".txt")).write_text("\n".join(linhas_novas))

# Open Images (já vem com train/ e val/ separados e um data.yaml próprio)
for split, pasta in [("train", OI_EXPORT_DIR / "train"), ("val", OI_EXPORT_DIR / "val")]:
    yaml_path = pasta / "dataset.yaml"
    if yaml_path.exists():
        classes_oi = yaml.safe_load(yaml_path.read_text())["names"]
        imgs = sorted((pasta / "images").glob("*.*"))
        copiar_e_remapear(imgs, split, classes_oi, pasta)

# Grocery Store + Freiburg (Roboflow já entrega train/valid/test com data.yaml próprio)
for nome_pasta in ["grocery_store", "freiburg_groceries"]:
    base = DATA_DIR / nome_pasta
    yaml_path = base / "data.yaml"
    if yaml_path.exists():
        classes_rf = yaml.safe_load(yaml_path.read_text())["names"]
        for split_origem, split_destino in [("train", "train"), ("valid", "val")]:
            pasta_split = base / split_origem
            if (pasta_split / "images").exists():
                imgs = sorted((pasta_split / "images").glob("*.*"))
                copiar_e_remapear(imgs, split_destino, classes_rf, pasta_split)

# Food-101 e Fruits-360 (bbox de imagem inteira, sem split — fazemos o split aqui)
for nome_pasta, classes_orig in [("food101_yolo", food101_raw.classes), ("fruits360_yolo", classes_fruits)]:
    base = DATA_DIR / nome_pasta
    if (base / "images").exists():
        for lista_imgs, split, cls, origem in registrar_fonte_yolo_padrao(base, classes_orig):
            copiar_e_remapear(lista_imgs, split, cls, origem)

# Escreve o data.yaml final
data_yaml_final = {
    "path": str(MERGED_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {i: nome for i, nome in enumerate(classes_finais)},
}
(MERGED_DIR / "data.yaml").write_text(yaml.dump(data_yaml_final, allow_unicode=True, sort_keys=False))

print(f"Dataset combinado pronto: {len(classes_finais)} classes únicas.")
print("data.yaml salvo em:", MERGED_DIR / "data.yaml")


## 3. Treinamento do modelo YOLO

Com os 5 datasets baixados e combinados na Seção 2, o treinamento roda direto — usa
`yolov8n.pt` (nano, rápido) como ponto de partida (transfer learning) e faz *fine-tuning*
no dataset combinado de alimentos.


In [ ]:
from ultralytics import YOLO

# Carrega um YOLOv8 nano pré-treinado no COCO como base (transfer learning)
modelo = YOLO("yolov8n.pt")

resultados = modelo.train(
    data=CAMINHO_DATA_YAML,     # data/merged/data.yaml, gerado na Seção 2.5
    epochs=50,
    imgsz=640,
    batch=16,
    project=str(MODELS_DIR),
    name="nutrivision_yolo",
    patience=10,          # early stopping
    verbose=True,
)

# Validação
metrics = modelo.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)

# Caminho do melhor checkpoint treinado — o app.py (Seção 5) já procura este caminho automaticamente
MODELO_TREINADO = MODELS_DIR / "nutrivision_yolo" / "weights" / "best.pt"
print("Modelo treinado salvo em:", MODELO_TREINADO)


Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/nutrivision/data/merged/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=nutrivision_yolo, nb

##4. Banco de dados nutricional (cache local + base própria)

Uso do SQLite como cache para não recalcular a cada detecção repetida, e uma base nutricional própria, com foco em alimentos brasileiros e itens comuns de supermercado (sem depender de API externa). Os valores são por 100g e baseados em referências nutricionais conhecidas — boas para cálculos aproximados de dieta.


In [ ]:
import sqlite3

def init_db():
    """Cria a tabela de cache nutricional se ainda não existir."""
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS nutrition_cache (
            food_name TEXT PRIMARY KEY,
            calories REAL,
            protein_g REAL,
            fat_g REAL,
            carbs_g REAL,
            fiber_g REAL,
            sodium_mg REAL,
            extra_json TEXT,
            source TEXT,
            updated_at TEXT
        )
    """)
    con.commit()
    con.close()

init_db()
print("Banco de dados nutricional pronto em:", DB_PATH)


Banco de dados nutricional pronto em: /content/nutrivision/nutrition_cache.db


In [ ]:
import sqlite3
from datetime import datetime

# Base nutricional local (por 100g) — foco em alimentos brasileiros e itens de mercado
BASE_LOCAL_ALIMENTOS = {
    # ---------------- FRUTAS ----------------
    "maca":              {"calories": 52,  "protein_g": 0.3, "fat_g": 0.2, "carbs_g": 14,  "fiber_g": 2.4, "sodium_mg": 1},
    "banana":            {"calories": 89,  "protein_g": 1.1, "fat_g": 0.3, "carbs_g": 23,  "fiber_g": 2.6, "sodium_mg": 1},
    "banana_prata":      {"calories": 98,  "protein_g": 1.3, "fat_g": 0.1, "carbs_g": 26,  "fiber_g": 2.0, "sodium_mg": 1},
    "laranja":           {"calories": 47,  "protein_g": 0.9, "fat_g": 0.1, "carbs_g": 12,  "fiber_g": 2.4, "sodium_mg": 0},
    "mamao":             {"calories": 43,  "protein_g": 0.5, "fat_g": 0.3, "carbs_g": 11,  "fiber_g": 1.7, "sodium_mg": 8},
    "manga":             {"calories": 60,  "protein_g": 0.8, "fat_g": 0.4, "carbs_g": 15,  "fiber_g": 1.6, "sodium_mg": 1},
    "abacaxi":           {"calories": 50,  "protein_g": 0.5, "fat_g": 0.1, "carbs_g": 13,  "fiber_g": 1.4, "sodium_mg": 1},
    "uva":               {"calories": 69,  "protein_g": 0.7, "fat_g": 0.2, "carbs_g": 18,  "fiber_g": 0.9, "sodium_mg": 2},
    "melancia":          {"calories": 30,  "protein_g": 0.6, "fat_g": 0.2, "carbs_g": 8,   "fiber_g": 0.4, "sodium_mg": 1},
    "morango":           {"calories": 32,  "protein_g": 0.7, "fat_g": 0.3, "carbs_g": 8,   "fiber_g": 2.0, "sodium_mg": 1},
    "abacate":           {"calories": 160, "protein_g": 2.0, "fat_g": 15,  "carbs_g": 9,   "fiber_g": 7.0, "sodium_mg": 7},
    "limao":             {"calories": 29,  "protein_g": 1.1, "fat_g": 0.3, "carbs_g": 9,   "fiber_g": 2.8, "sodium_mg": 2},
    "tangerina":         {"calories": 53,  "protein_g": 0.8, "fat_g": 0.3, "carbs_g": 13,  "fiber_g": 1.8, "sodium_mg": 2},
    "pera":              {"calories": 57,  "protein_g": 0.4, "fat_g": 0.1, "carbs_g": 15,  "fiber_g": 3.1, "sodium_mg": 1},
    "goiaba":            {"calories": 68,  "protein_g": 2.6, "fat_g": 1.0, "carbs_g": 14,  "fiber_g": 5.4, "sodium_mg": 2},
    "maracuja":          {"calories": 68,  "protein_g": 2.2, "fat_g": 0.7, "carbs_g": 12,  "fiber_g": 10,  "sodium_mg": 28},
    "acerola":           {"calories": 32,  "protein_g": 0.4, "fat_g": 0.3, "carbs_g": 8,   "fiber_g": 1.1, "sodium_mg": 7},
    "caju":              {"calories": 43,  "protein_g": 0.9, "fat_g": 0.3, "carbs_g": 11,  "fiber_g": 1.7, "sodium_mg": 2},
    "coco_polpa":        {"calories": 354, "protein_g": 3.3, "fat_g": 33,  "carbs_g": 15,  "fiber_g": 9.0, "sodium_mg": 20},

    # ---------------- VERDURAS E LEGUMES ----------------
    "tomate":            {"calories": 18,  "protein_g": 0.9, "fat_g": 0.2, "carbs_g": 3.9, "fiber_g": 1.2, "sodium_mg": 5},
    "alface":            {"calories": 15,  "protein_g": 1.4, "fat_g": 0.2, "carbs_g": 2.9, "fiber_g": 1.3, "sodium_mg": 28},
    "cenoura":           {"calories": 41,  "protein_g": 0.9, "fat_g": 0.2, "carbs_g": 10,  "fiber_g": 2.8, "sodium_mg": 69},
    "batata":            {"calories": 77,  "protein_g": 2.0, "fat_g": 0.1, "carbs_g": 17,  "fiber_g": 2.2, "sodium_mg": 6},
    "batata_doce":       {"calories": 86,  "protein_g": 1.6, "fat_g": 0.1, "carbs_g": 20,  "fiber_g": 3.0, "sodium_mg": 55},
    "mandioca":          {"calories": 160, "protein_g": 1.4, "fat_g": 0.3, "carbs_g": 38,  "fiber_g": 1.8, "sodium_mg": 14},
    "inhame":            {"calories": 118, "protein_g": 1.5, "fat_g": 0.2, "carbs_g": 28,  "fiber_g": 4.1, "sodium_mg": 9},
    "cebola":            {"calories": 40,  "protein_g": 1.1, "fat_g": 0.1, "carbs_g": 9.3, "fiber_g": 1.7, "sodium_mg": 4},
    "alho":              {"calories": 149, "protein_g": 6.4, "fat_g": 0.5, "carbs_g": 33,  "fiber_g": 2.1, "sodium_mg": 17},
    "pepino":            {"calories": 15,  "protein_g": 0.7, "fat_g": 0.1, "carbs_g": 3.6, "fiber_g": 0.5, "sodium_mg": 2},
    "abobrinha":         {"calories": 17,  "protein_g": 1.2, "fat_g": 0.3, "carbs_g": 3.1, "fiber_g": 1.0, "sodium_mg": 8},
    "abobora":           {"calories": 26,  "protein_g": 1.0, "fat_g": 0.1, "carbs_g": 6.5, "fiber_g": 0.5, "sodium_mg": 1},
    "brocolis":          {"calories": 34,  "protein_g": 2.8, "fat_g": 0.4, "carbs_g": 7.0, "fiber_g": 2.6, "sodium_mg": 33},
    "couve":             {"calories": 27,  "protein_g": 2.9, "fat_g": 0.5, "carbs_g": 4.3, "fiber_g": 2.3, "sodium_mg": 15},
    "espinafre":         {"calories": 23,  "protein_g": 2.9, "fat_g": 0.4, "carbs_g": 3.6, "fiber_g": 2.2, "sodium_mg": 79},
    "pimentao":          {"calories": 31,  "protein_g": 1.0, "fat_g": 0.3, "carbs_g": 6.0, "fiber_g": 2.1, "sodium_mg": 4},
    "beterraba":         {"calories": 43,  "protein_g": 1.6, "fat_g": 0.2, "carbs_g": 10,  "fiber_g": 2.8, "sodium_mg": 78},
    "chuchu":            {"calories": 19,  "protein_g": 0.8, "fat_g": 0.1, "carbs_g": 4.5, "fiber_g": 1.7, "sodium_mg": 2},
    "vagem":             {"calories": 31,  "protein_g": 1.8, "fat_g": 0.1, "carbs_g": 7.0, "fiber_g": 3.4, "sodium_mg": 6},
    "quiabo":            {"calories": 33,  "protein_g": 1.9, "fat_g": 0.2, "carbs_g": 7.5, "fiber_g": 3.2, "sodium_mg": 7},
    "milho_verde":       {"calories": 96,  "protein_g": 3.4, "fat_g": 1.5, "carbs_g": 19,  "fiber_g": 2.7, "sodium_mg": 15},

    # ---------------- GRÃOS, CEREAIS E MASSAS ----------------
    "arroz":             {"calories": 130, "protein_g": 2.7, "fat_g": 0.3, "carbs_g": 28,  "fiber_g": 0.4, "sodium_mg": 1},
    "arroz_integral":    {"calories": 123, "protein_g": 2.6, "fat_g": 1.0, "carbs_g": 26,  "fiber_g": 1.8, "sodium_mg": 4},
    "feijao":            {"calories": 127, "protein_g": 8.7, "fat_g": 0.5, "carbs_g": 23,  "fiber_g": 6.4, "sodium_mg": 2},
    "feijao_preto":      {"calories": 132, "protein_g": 8.9, "fat_g": 0.5, "carbs_g": 24,  "fiber_g": 8.7, "sodium_mg": 2},
    "lentilha":          {"calories": 116, "protein_g": 9.0, "fat_g": 0.4, "carbs_g": 20,  "fiber_g": 7.9, "sodium_mg": 2},
    "grao_de_bico":      {"calories": 164, "protein_g": 8.9, "fat_g": 2.6, "carbs_g": 27,  "fiber_g": 7.6, "sodium_mg": 7},
    "aveia":             {"calories": 389, "protein_g": 17,  "fat_g": 7.0, "carbs_g": 66,  "fiber_g": 10.6,"sodium_mg": 2},
    "quinoa":            {"calories": 120, "protein_g": 4.4, "fat_g": 1.9, "carbs_g": 21,  "fiber_g": 2.8, "sodium_mg": 7},
    "macarrao":          {"calories": 158, "protein_g": 5.8, "fat_g": 0.9, "carbs_g": 31,  "fiber_g": 1.8, "sodium_mg": 6},
    "farinha_de_trigo":  {"calories": 364, "protein_g": 10,  "fat_g": 1.0, "carbs_g": 76,  "fiber_g": 2.7, "sodium_mg": 2},
    "farinha_de_mandioca":{"calories": 361,"protein_g": 1.6, "fat_g": 0.3, "carbs_g": 88,  "fiber_g": 6.4, "sodium_mg": 6},
    "fuba":              {"calories": 365, "protein_g": 8.1, "fat_g": 3.9, "carbs_g": 77,  "fiber_g": 7.3, "sodium_mg": 4},
    "tapioca_goma":      {"calories": 240, "protein_g": 0.2, "fat_g": 0.0, "carbs_g": 59,  "fiber_g": 0.5, "sodium_mg": 5},
    "granola":           {"calories": 471, "protein_g": 10,  "fat_g": 20,  "carbs_g": 64,  "fiber_g": 7.0, "sodium_mg": 90},

    # ---------------- PÃES E PADARIA ----------------
    "pao_frances":       {"calories": 300, "protein_g": 9.4, "fat_g": 3.1, "carbs_g": 58,  "fiber_g": 2.3, "sodium_mg": 540},
    "pao_de_forma":      {"calories": 265, "protein_g": 9.0, "fat_g": 3.3, "carbs_g": 49,  "fiber_g": 2.4, "sodium_mg": 490},
    "pao_integral":      {"calories": 246, "protein_g": 10,  "fat_g": 3.4, "carbs_g": 41,  "fiber_g": 6.0, "sodium_mg": 400},
    "pao_de_queijo":     {"calories": 320, "protein_g": 6.0, "fat_g": 18,  "carbs_g": 33,  "fiber_g": 0.8, "sodium_mg": 400},
    "pao_doce":          {"calories": 310, "protein_g": 7.5, "fat_g": 7.0, "carbs_g": 55,  "fiber_g": 1.5, "sodium_mg": 300},

    # ---------------- OVOS, CARNES E PEIXES ----------------
    "ovo":               {"calories": 155, "protein_g": 13,  "fat_g": 11,  "carbs_g": 1.1, "fiber_g": 0,   "sodium_mg": 124},
    "frango_peito":      {"calories": 165, "protein_g": 31,  "fat_g": 3.6, "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 74},
    "frango_coxa":       {"calories": 209, "protein_g": 26,  "fat_g": 11,  "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 90},
    "carne_bovina_moida":{"calories": 215, "protein_g": 26,  "fat_g": 12,  "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 66},
    "carne_bovina_file": {"calories": 187, "protein_g": 27,  "fat_g": 8.0, "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 58},
    "picanha":           {"calories": 259, "protein_g": 24,  "fat_g": 17,  "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 60},
    "carne_de_porco":    {"calories": 242, "protein_g": 27,  "fat_g": 14,  "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 62},
    "linguica":          {"calories": 301, "protein_g": 13,  "fat_g": 27,  "carbs_g": 1.5, "fiber_g": 0,   "sodium_mg": 850},
    "bacon":             {"calories": 541, "protein_g": 37,  "fat_g": 42,  "carbs_g": 1.4, "fiber_g": 0,   "sodium_mg": 1717},
    "presunto":          {"calories": 145, "protein_g": 18,  "fat_g": 6.0, "carbs_g": 2.0, "fiber_g": 0,   "sodium_mg": 1200},
    "mortadela":         {"calories": 311, "protein_g": 14,  "fat_g": 27,  "carbs_g": 2.5, "fiber_g": 0,   "sodium_mg": 1050},
    "salame":            {"calories": 336, "protein_g": 22,  "fat_g": 26,  "carbs_g": 1.9, "fiber_g": 0,   "sodium_mg": 1600},
    "peixe_tilapia":     {"calories": 96,  "protein_g": 20,  "fat_g": 1.7, "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 52},
    "sardinha_lata":     {"calories": 208, "protein_g": 25,  "fat_g": 11,  "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 400},
    "atum_lata":         {"calories": 116, "protein_g": 26,  "fat_g": 1.0, "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 300},
    "camarao":           {"calories": 99,  "protein_g": 24,  "fat_g": 0.3, "carbs_g": 0.2, "fiber_g": 0,   "sodium_mg": 111},

    # ---------------- LATICÍNIOS ----------------
    "leite_caixa":       {"calories": 61,  "protein_g": 3.2, "fat_g": 3.3, "carbs_g": 4.8, "fiber_g": 0,   "sodium_mg": 43},
    "leite_integral":    {"calories": 61,  "protein_g": 3.2, "fat_g": 3.3, "carbs_g": 4.8, "fiber_g": 0,   "sodium_mg": 43},
    "leite_desnatado":   {"calories": 35,  "protein_g": 3.4, "fat_g": 0.1, "carbs_g": 5.0, "fiber_g": 0,   "sodium_mg": 42},
    "iogurte":           {"calories": 59,  "protein_g": 10,  "fat_g": 0.4, "carbs_g": 3.6, "fiber_g": 0,   "sodium_mg": 36},
    "iogurte_grego":     {"calories": 97,  "protein_g": 9.0, "fat_g": 5.0, "carbs_g": 3.9, "fiber_g": 0,   "sodium_mg": 35},
    "queijo_minas":      {"calories": 264, "protein_g": 17,  "fat_g": 21,  "carbs_g": 3.0, "fiber_g": 0,   "sodium_mg": 400},
    "queijo_mussarela":  {"calories": 280, "protein_g": 22,  "fat_g": 21,  "carbs_g": 2.2, "fiber_g": 0,   "sodium_mg": 620},
    "requeijao":         {"calories": 264, "protein_g": 9.0, "fat_g": 24,  "carbs_g": 3.0, "fiber_g": 0,   "sodium_mg": 450},
    "manteiga":          {"calories": 717, "protein_g": 0.9, "fat_g": 81,  "carbs_g": 0.1, "fiber_g": 0,   "sodium_mg": 11},
    "margarina":         {"calories": 596, "protein_g": 0.2, "fat_g": 66,  "carbs_g": 1.0, "fiber_g": 0,   "sodium_mg": 800},

    # ---------------- ÓLEOS E GORDURAS ----------------
    "azeite":            {"calories": 884, "protein_g": 0,   "fat_g": 100, "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 2},
    "oleo_soja":         {"calories": 884, "protein_g": 0,   "fat_g": 100, "carbs_g": 0,   "fiber_g": 0,   "sodium_mg": 0},

    # ---------------- INDUSTRIALIZADOS / MERCADO ----------------
    "biscoito_maisena":  {"calories": 443, "protein_g": 7.5, "fat_g": 13,  "carbs_g": 74,  "fiber_g": 1.8, "sodium_mg": 320},
    "biscoito_recheado": {"calories": 480, "protein_g": 5.0, "fat_g": 20,  "carbs_g": 70,  "fiber_g": 1.5, "sodium_mg": 250},
    "achocolatado_po":   {"calories": 385, "protein_g": 4.5, "fat_g": 3.0, "carbs_g": 85,  "fiber_g": 2.5, "sodium_mg": 130},
    "refrigerante":      {"calories": 42,  "protein_g": 0,   "fat_g": 0,   "carbs_g": 10.6,"fiber_g": 0,   "sodium_mg": 5},
    "suco_caixa":        {"calories": 45,  "protein_g": 0.2, "fat_g": 0,   "carbs_g": 11,  "fiber_g": 0.2, "sodium_mg": 8},
    "salgadinho_batata": {"calories": 536, "protein_g": 6.6, "fat_g": 35,  "carbs_g": 50,  "fiber_g": 4.4, "sodium_mg": 550},
    "macarrao_instantaneo":{"calories": 436,"protein_g": 9.0,"fat_g": 17,  "carbs_g": 62,  "fiber_g": 2.0, "sodium_mg": 1700},
    "molho_de_tomate":   {"calories": 32,  "protein_g": 1.6, "fat_g": 0.3, "carbs_g": 7.0, "fiber_g": 1.5, "sodium_mg": 380},
    "maionese":          {"calories": 680, "protein_g": 1.0, "fat_g": 75,  "carbs_g": 1.0, "fiber_g": 0,   "sodium_mg": 550},
    "ketchup":           {"calories": 112, "protein_g": 1.2, "fat_g": 0.2, "carbs_g": 27,  "fiber_g": 0.4, "sodium_mg": 900},
    "mostarda":          {"calories": 66,  "protein_g": 4.0, "fat_g": 4.0, "carbs_g": 5.0, "fiber_g": 1.0, "sodium_mg": 1100},
    "acucar":            {"calories": 387, "protein_g": 0,   "fat_g": 0,   "carbs_g": 100, "fiber_g": 0,   "sodium_mg": 1},
    "mel":               {"calories": 304, "protein_g": 0.3, "fat_g": 0,   "carbs_g": 82,  "fiber_g": 0.2, "sodium_mg": 4},
    "chocolate_ao_leite":{"calories": 535, "protein_g": 7.7, "fat_g": 30,  "carbs_g": 59,  "fiber_g": 3.4, "sodium_mg": 79},
    "cafe_po":           {"calories": 2,   "protein_g": 0.1, "fat_g": 0.0, "carbs_g": 0.3, "fiber_g": 0,   "sodium_mg": 2},
    "pizza_congelada":   {"calories": 266, "protein_g": 11,  "fat_g": 10,  "carbs_g": 33,  "fiber_g": 2.0, "sodium_mg": 600},
    "hamburguer_congelado":{"calories": 250,"protein_g": 14, "fat_g": 20,  "carbs_g": 3.0, "fiber_g": 0.5, "sodium_mg": 480},
    "nuggets_congelado": {"calories": 296, "protein_g": 15,  "fat_g": 18,  "carbs_g": 18,  "fiber_g": 1.0, "sodium_mg": 550},
    "sorvete":           {"calories": 207, "protein_g": 3.5, "fat_g": 11,  "carbs_g": 24,  "fiber_g": 0.7, "sodium_mg": 80},
    "gelatina_pronta":   {"calories": 62,  "protein_g": 1.5, "fat_g": 0,   "carbs_g": 14,  "fiber_g": 0,   "sodium_mg": 50},

    # ---------------- SALGADOS E DOCES TÍPICOS ----------------
    "coxinha":           {"calories": 274, "protein_g": 10,  "fat_g": 15,  "carbs_g": 25,  "fiber_g": 1.2, "sodium_mg": 480},
    "pastel":            {"calories": 290, "protein_g": 6.0, "fat_g": 18,  "carbs_g": 27,  "fiber_g": 1.0, "sodium_mg": 400},
    "brigadeiro":        {"calories": 411, "protein_g": 5.0, "fat_g": 14,  "carbs_g": 68,  "fiber_g": 1.0, "sodium_mg": 60},
    "pao_de_alho":       {"calories": 330, "protein_g": 7.0, "fat_g": 16,  "carbs_g": 40,  "fiber_g": 2.0, "sodium_mg": 520},
    "farofa_pronta":     {"calories": 411, "protein_g": 3.0, "fat_g": 15,  "carbs_g": 65,  "fiber_g": 5.0, "sodium_mg": 700},

    # ---------------- OLEAGINOSAS ----------------
    "castanha_de_caju":  {"calories": 553, "protein_g": 18,  "fat_g": 44,  "carbs_g": 30,  "fiber_g": 3.3, "sodium_mg": 12},
    "castanha_do_para":  {"calories": 656, "protein_g": 14,  "fat_g": 66,  "carbs_g": 12,  "fiber_g": 7.5, "sodium_mg": 3},
    "amendoim":          {"calories": 567, "protein_g": 26,  "fat_g": 49,  "carbs_g": 16,  "fiber_g": 8.5, "sodium_mg": 18},
}


def obter_info_nutricional(nome_alimento: str, peso_g: float = 100.0):
    """Retorna macro/micronutrientes para `peso_g` gramas do alimento.
    Ordem de busca: cache SQLite -> base local -> desconhecido."""
    nome_alimento = nome_alimento.lower().strip()

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("SELECT calories, protein_g, fat_g, carbs_g, fiber_g, sodium_mg, source FROM nutrition_cache WHERE food_name = ?", (nome_alimento,))
    linha = cur.fetchone()

    if linha is None:
        dados = BASE_LOCAL_ALIMENTOS.get(nome_alimento)
        if dados is None:
            con.close()
            return None
        fonte = "base local (Brasil)"
        cur.execute("""
            INSERT OR REPLACE INTO nutrition_cache
            (food_name, calories, protein_g, fat_g, carbs_g, fiber_g, sodium_mg, extra_json, source, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            nome_alimento, dados["calories"], dados["protein_g"], dados["fat_g"],
            dados["carbs_g"], dados["fiber_g"], dados["sodium_mg"],
            "{}", fonte, datetime.utcnow().isoformat(),
        ))
        con.commit()
        linha = (dados["calories"], dados["protein_g"], dados["fat_g"], dados["carbs_g"], dados["fiber_g"], dados["sodium_mg"], fonte)

    con.close()
    fator = peso_g / 100.0
    calorias, proteina, gordura, carbo, fibra, sodio, fonte = linha
    return {
        "peso_g": peso_g,
        "calories": round(calorias * fator, 1),
        "protein_g": round(proteina * fator, 1),
        "fat_g": round(gordura * fator, 1),
        "carbs_g": round(carbo * fator, 1),
        "fiber_g": round(fibra * fator, 1),
        "sodium_mg": round(sodio * fator, 1),
        "source": fonte,
    }

# Teste rápido
print(obter_info_nutricional("banana", peso_g=150))

{'peso_g': 150, 'calories': 133.5, 'protein_g': 1.7, 'fat_g': 0.4, 'carbs_g': 34.5, 'fiber_g': 3.9, 'sodium_mg': 1.5, 'source': 'base local (Brasil)'}


datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).


## 5. Interface Streamlit — NutriVision

Gera o arquivo `app.py` com a interface completa: 3 modos de entrada (câmera, foto, vídeo),
detecção YOLO com informações nutricionais e
histórico de alimentos detectados na sessão.


In [ ]:
%%writefile /content/nutrivision/app.py
import streamlit as st
from ultralytics import YOLO
from PIL import Image
import numpy as np
import cv2
import tempfile
import sqlite3
import json
import os
from pathlib import Path
from datetime import datetime

# --------------------------------------------------------------------------
# CONFIGURAÇÃO DE PÁGINA
# --------------------------------------------------------------------------
st.set_page_config(page_title="NutriVision", page_icon="🥗", layout="wide")

PROJECT_DIR = Path("/content/nutrivision")
DB_PATH = PROJECT_DIR / "nutrition_cache.db"
# Caminho do modelo: usa o treinado se existir, senão cai no YOLOv8n padrão (placeholder)
MODELO_TREINADO = PROJECT_DIR / "models" / "nutrivision_yolo" / "weights" / "best.pt"
CAMINHO_MODELO = str(MODELO_TREINADO) if MODELO_TREINADO.exists() else "yolov8n.pt"

# --------------------------------------------------------------------------
# CSS CUSTOMIZADO — paleta pastel + destaques vibrantes, cards arredondados
# --------------------------------------------------------------------------
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&family=Playfair+Display:ital@1&display=swap');

html, body, [class*="css"]  { font-family: 'Poppins', sans-serif; }

.stApp {
    background: radial-gradient(circle at 10% 10%, #FBE9E7 0%, #F5EFE6 35%, #EFE7DA 100%);
}

.nv-title {
    font-family: 'Playfair Display', serif;
    font-style: italic;
    font-size: 2.6rem;
    color: #1F3D2B;
    margin-bottom: 0;
}
.nv-subtitle {
    font-family: 'Poppins', sans-serif;
    color: #4A4A4A;
    font-size: 1rem;
    margin-top: 0.2rem;
}

.nv-card {
    background: #FFFFFF;
    border-radius: 22px;
    padding: 1.4rem 1.6rem;
    box-shadow: 0 8px 24px rgba(31, 61, 43, 0.08);
    margin-bottom: 1rem;
}
.nv-card-accent {
    background: #1F3D2B;
    color: #F5EFE6;
    border-radius: 22px;
    padding: 1.4rem 1.6rem;
    box-shadow: 0 8px 24px rgba(31, 61, 43, 0.15);
}

.nv-pill {
    display: inline-block;
    padding: 0.25rem 0.8rem;
    border-radius: 999px;
    font-size: 0.8rem;
    font-weight: 600;
    margin-right: 0.4rem;
}
.nv-pill-lime   { background: #D8ED6F; color: #1F3D2B; }
.nv-pill-mustard{ background: #E8B84B; color: #3B2A00; }
.nv-pill-teal   { background: #2A7F7E; color: #FFFFFF; }

.nv-nutrient-row {
    display: flex;
    justify-content: space-between;
    padding: 0.35rem 0;
    border-bottom: 1px dashed #E3DACB;
    font-size: 0.95rem;
}
.nv-nutrient-row:last-child { border-bottom: none; }

.stButton>button {
    background-color: #1F3D2B;
    color: #F5EFE6;
    border-radius: 999px;
    border: none;
    padding: 0.5rem 1.4rem;
    font-weight: 600;
}
.stButton>button:hover { background-color: #2A7F7E; color: white; }

section[data-testid="stSidebar"] {
    background: #F8F2E9;
}
</style>
""", unsafe_allow_html=True)

# --------------------------------------------------------------------------
# CABEÇALHO
# --------------------------------------------------------------------------
col_logo, col_title = st.columns([1, 6])
with col_logo:
    st.markdown("### 🥗")
with col_title:
    st.markdown('<p class="nv-title">NutriVision</p>', unsafe_allow_html=True)
    st.markdown('<p class="nv-subtitle">Identifique alimentos e descubra sua nutrição em segundos</p>', unsafe_allow_html=True)

st.write("")

# --------------------------------------------------------------------------
# CARREGAMENTO DO MODELO (cacheado para não recarregar a cada interação)
# --------------------------------------------------------------------------
@st.cache_resource
def carregar_modelo(caminho):
    return YOLO(caminho)

modelo = carregar_modelo(CAMINHO_MODELO)

if CAMINHO_MODELO == "yolov8n.pt":
    st.markdown(
        '<div class="nv-card" style="border-left:6px solid #E8B84B;">'
        '⚠️ Usando o YOLOv8 pré-treinado no COCO como placeholder (classes genéricas). '
        'Treine o modelo especializado em alimentos na Seção 3 do notebook para resultados melhores.'
        '</div>', unsafe_allow_html=True
    )

# --------------------------------------------------------------------------
# FUNÇÕES NUTRICIONAIS (mesma lógica da Seção 4 do notebook)
# --------------------------------------------------------------------------
BASE_LOCAL_ALIMENTOS = {
    "maca": {"calories": 52, "protein_g": 0.3, "fat_g": 0.2, "carbs_g": 14, "fiber_g": 2.4, "sodium_mg": 1},
    "banana": {"calories": 89, "protein_g": 1.1, "fat_g": 0.3, "carbs_g": 23, "fiber_g": 2.6, "sodium_mg": 1},
    "apple": {"calories": 52, "protein_g": 0.3, "fat_g": 0.2, "carbs_g": 14, "fiber_g": 2.4, "sodium_mg": 1},
    "orange": {"calories": 47, "protein_g": 0.9, "fat_g": 0.1, "carbs_g": 12, "fiber_g": 2.4, "sodium_mg": 0},
    "sandwich": {"calories": 250, "protein_g": 10, "fat_g": 9, "carbs_g": 30, "fiber_g": 2, "sodium_mg": 480},
    "pizza": {"calories": 266, "protein_g": 11, "fat_g": 10, "carbs_g": 33, "fiber_g": 2.3, "sodium_mg": 598},
    "donut": {"calories": 452, "protein_g": 4.9, "fat_g": 25, "carbs_g": 51, "fiber_g": 1.5, "sodium_mg": 373},
    "cake": {"calories": 350, "protein_g": 5, "fat_g": 15, "carbs_g": 50, "fiber_g": 1, "sodium_mg": 300},
    "carrot": {"calories": 41, "protein_g": 0.9, "fat_g": 0.2, "carbs_g": 10, "fiber_g": 2.8, "sodium_mg": 69},
    "broccoli": {"calories": 34, "protein_g": 2.8, "fat_g": 0.4, "carbs_g": 7, "fiber_g": 2.6, "sodium_mg": 33},
}

def init_db():
    con = sqlite3.connect(DB_PATH)
    con.execute("""
        CREATE TABLE IF NOT EXISTS nutrition_cache (
            food_name TEXT PRIMARY KEY, calories REAL, protein_g REAL, fat_g REAL,
            carbs_g REAL, fiber_g REAL, sodium_mg REAL, extra_json TEXT, source TEXT, updated_at TEXT
        )
    """)
    con.commit()
    con.close()

init_db()

def obter_info_nutricional(nome_alimento: str, peso_g: float = 100.0):
    nome_alimento = nome_alimento.lower().strip()
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("SELECT calories, protein_g, fat_g, carbs_g, fiber_g, sodium_mg, source FROM nutrition_cache WHERE food_name = ?", (nome_alimento,))
    linha = cur.fetchone()
    if linha is None:
        dados = BASE_LOCAL_ALIMENTOS.get(nome_alimento)
        if dados is None:
            con.close()
            return None
        cur.execute("""INSERT OR REPLACE INTO nutrition_cache
            (food_name, calories, protein_g, fat_g, carbs_g, fiber_g, sodium_mg, extra_json, source, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
            (nome_alimento, dados["calories"], dados["protein_g"], dados["fat_g"], dados["carbs_g"],
             dados["fiber_g"], dados["sodium_mg"], "{}", "base local", datetime.utcnow().isoformat()))
        con.commit()
        linha = (dados["calories"], dados["protein_g"], dados["fat_g"], dados["carbs_g"], dados["fiber_g"], dados["sodium_mg"], "base local")
    con.close()
    fator = peso_g / 100.0
    calorias, proteina, gordura, carbo, fibra, sodio, fonte = linha
    return {"peso_g": peso_g, "calories": round(calorias * fator, 1), "protein_g": round(proteina * fator, 1),
            "fat_g": round(gordura * fator, 1), "carbs_g": round(carbo * fator, 1),
            "fiber_g": round(fibra * fator, 1), "sodium_mg": round(sodio * fator, 1), "source": fonte}

# --------------------------------------------------------------------------
# ESTADO DA SESSÃO (histórico)
# --------------------------------------------------------------------------
if "historico" not in st.session_state:
    st.session_state.historico = []

# --------------------------------------------------------------------------
# SIDEBAR — modo de entrada e peso estimado
# --------------------------------------------------------------------------
with st.sidebar:
    st.markdown("### 📷 Modo de entrada")
    modo = st.radio("Escolha como identificar o alimento:", ["Câmera", "Foto (upload)", "Vídeo (upload)"], label_visibility="collapsed")
    st.markdown("---")
    peso_estimado = st.slider("Peso estimado da porção (g)", 10, 500, 100, step=10)
    confianca_min = st.slider("Confiança mínima de detecção", 0.1, 0.9, 0.35, step=0.05)
    st.markdown("---")
    st.markdown("### 🕘 Histórico da sessão")
    if st.session_state.historico:
        for item in reversed(st.session_state.historico[-8:]):
            st.markdown(f"- **{item['nome']}** · {item['calorias']} kcal")
    else:
        st.caption("Nenhum alimento detectado ainda.")

# --------------------------------------------------------------------------
# FUNÇÃO DE DETECÇÃO + RENDERIZAÇÃO DE RESULTADOS
# --------------------------------------------------------------------------
def detectar_e_mostrar(imagem_np, conf):
    resultados = modelo.predict(imagem_np, conf=conf, verbose=False)
    r = resultados[0]
    imagem_anotada = r.plot()  # imagem com bounding boxes desenhadas (BGR)
    imagem_anotada_rgb = cv2.cvtColor(imagem_anotada, cv2.COLOR_BGR2RGB)

    col_img, col_info = st.columns([3, 2])
    with col_img:
        st.markdown('<div class="nv-card">', unsafe_allow_html=True)
        st.image(imagem_anotada_rgb, use_column_width=True, caption="Detecção em tempo real")
        st.markdown('</div>', unsafe_allow_html=True)

    with col_info:
        if len(r.boxes) == 0:
            st.markdown('<div class="nv-card">Nenhum alimento detectado. Tente aproximar a câmera ou melhorar a iluminação.</div>', unsafe_allow_html=True)
            return

        nomes_detectados = {}
        for box in r.boxes:
            nome_classe = modelo.names[int(box.cls[0])]
            confianca = float(box.conf[0])
            if nome_classe not in nomes_detectados or confianca > nomes_detectados[nome_classe]:
                nomes_detectados[nome_classe] = confianca

        for nome_classe, confianca in nomes_detectados.items():
            info = obter_info_nutricional(nome_classe, peso_g=peso_estimado)
            st.markdown('<div class="nv-card">', unsafe_allow_html=True)
            st.markdown(
                f'<span class="nv-pill nv-pill-lime">{nome_classe.capitalize()}</span>'
                f'<span class="nv-pill nv-pill-teal">Confiança {confianca*100:.0f}%</span>'
                f'<span class="nv-pill nv-pill-mustard">{peso_estimado} g</span>',
                unsafe_allow_html=True,
            )
            st.write("")
            if info is None:
                st.warning("Sem dados nutricionais cadastrados para este alimento ainda.")
            else:
                st.markdown(f"""
                    <div class="nv-nutrient-row"><b>Calorias</b><span>{info['calories']} kcal</span></div>
                    <div class="nv-nutrient-row"><b>Proteínas</b><span>{info['protein_g']} g</span></div>
                    <div class="nv-nutrient-row"><b>Gorduras</b><span>{info['fat_g']} g</span></div>
                    <div class="nv-nutrient-row"><b>Carboidratos</b><span>{info['carbs_g']} g</span></div>
                    <div class="nv-nutrient-row"><b>Fibras</b><span>{info['fiber_g']} g</span></div>
                    <div class="nv-nutrient-row"><b>Sódio</b><span>{info['sodium_mg']} mg</span></div>
                    <p style="font-size:0.75rem;color:#888;margin-top:0.6rem;">Fonte: {info['source']}</p>
                """, unsafe_allow_html=True)
                st.session_state.historico.append({"nome": nome_classe, "calorias": info["calories"], "hora": datetime.now().strftime("%H:%M:%S")})
            st.markdown('</div>', unsafe_allow_html=True)

# --------------------------------------------------------------------------
# MODOS DE ENTRADA
# --------------------------------------------------------------------------
if modo == "Câmera":
    st.markdown('<div class="nv-card-accent">📸 Tire uma foto do alimento com a câmera do dispositivo.</div>', unsafe_allow_html=True)
    foto = st.camera_input("Câmera", label_visibility="collapsed")
    if foto is not None:
        imagem = np.array(Image.open(foto).convert("RGB"))
        detectar_e_mostrar(imagem, confianca_min)

elif modo == "Foto (upload)":
    st.markdown('<div class="nv-card-accent">🖼️ Envie uma foto do alimento (JPG ou PNG).</div>', unsafe_allow_html=True)
    arquivo = st.file_uploader("Upload de foto", type=["jpg", "jpeg", "png"], label_visibility="collapsed")
    if arquivo is not None:
        imagem = np.array(Image.open(arquivo).convert("RGB"))
        detectar_e_mostrar(imagem, confianca_min)

else:  # Vídeo
    st.markdown('<div class="nv-card-accent">🎬 Envie um vídeo curto — analisamos um frame representativo.</div>', unsafe_allow_html=True)
    video_arquivo = st.file_uploader("Upload de vídeo", type=["mp4", "mov", "avi"], label_visibility="collapsed")
    if video_arquivo is not None:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as tmp:
            tmp.write(video_arquivo.read())
            caminho_video = tmp.name
        captura = cv2.VideoCapture(caminho_video)
        total_frames = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
        captura.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)  # pega o frame do meio
        ok, frame = captura.read()
        captura.release()
        if ok:
            imagem = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            detectar_e_mostrar(imagem, confianca_min)
        else:
            st.error("Não foi possível ler o vídeo enviado.")


Writing /content/nutrivision/app.py


## 6. Deploy — Streamlit Community Cloud (recomendado)
O app é públicado usando o repositório separado do projeto em [share.streamlit.io](https://share.streamlit.io)



## 6.1 Rodando o Streamlit dentro do Colab

O Colab não expõe portas diretamente, então foi utilizado o **localtunnel** para gerar uma URL pública
temporária. Rode a célula abaixo e clique no link gerado (a senha do túnel é o IP mostrado).


In [ ]:
import urllib.request
import time

# IP público do runtime do Colab — usado como "senha" do localtunnel
ip_publico = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("Senha do túnel (Tunnel Password):", ip_publico)

# Mata qualquer processo antigo do Streamlit/localtunnel que tenha ficado rodando
!pkill -f streamlit
!pkill -f localtunnel
time.sleep(2)

# Inicia o Streamlit em background, com flags que evitam bloqueios de CORS/XSRF
# e garantem modo headless (sem esperar input no terminal)
!streamlit run /content/nutrivision/app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    &> /content/logs.txt &

# Espera o Streamlit terminar de subir de fato antes de abrir o túnel
# (evita o túnel conectar a um servidor ainda incompleto)
print("Aguardando o Streamlit inicializar...")
time.sleep(15)

# Confirma no log se o servidor já está no ar
!grep -i "you can now view" /content/logs.txt || echo "Streamlit ainda subindo, aguarde mais alguns segundos..."

# Só agora abre o túnel público
!npx localtunnel --port 8501

Senha do túnel (Tunnel Password): 136.85.57.211
Aguardando o Streamlit inicializar...
  You can now view your Streamlit app in your browser.
⠙⠹⠸⠼your url is: https://whole-toys-float.loca.lt
^C
